### Testing Qwen-3.5:9B OCR and Vision Model Tasks

#### First step is to import the pipeline.py and process any PDFs to images

In [4]:
# import sys
# !{sys.executable} -m pip install pdf2image ollama


In [20]:
import importlib
import pipeline as pipeline_module
from pathlib import Path
import os
import pipeline as pipeline_module

importlib.reload(pipeline_module)
pipeline = pipeline_module.OCRPipeline()

# convert any existing .pdf files to .png for testing
# pages = []
# for file in os.listdir("pdf_images"):
#     if file.endswith(".pdf"):
#         pdf_path = os.path.join("pdf_images", file)
#         page_images = pipeline.convert_pdf_to_images(pdf_path)
#         pages.extend(page_images)

# print(f"Converted {len(pages)} page image(s)")


In [18]:
import ollama

response = ollama.chat(
    model="qwen3.5:9b",
    messages=[
        {
            "role": "user",
            "content": "Describe what is visible in this image in one sentence.",
            "images": ["images/PHYS207_PAGE_1_page_1.png"],
        }
    ],
    options={"num_predict": 100},
)

print(response)


model='qwen3.5:9b' created_at='2026-03-11T06:50:41.927313Z' done=True done_reason='length' total_duration=36012893708 load_duration=126159625 prompt_eval_count=2012 prompt_eval_duration=28072387625 eval_count=100 eval_duration=7549139707 message=Message(role='assistant', content='', thinking='The user wants a one-sentence description of the provided image.\n\n1.  **Identify the main subject:** It\'s a handwritten physics study note or worksheet.\n2.  **Identify the topic:** The title is "Electric Charge & Coulomb\'s Law".\n3.  **Identify the date:** Top left says "1/12/26" (likely a future date for a class, or just a generic date format, but probably means Jan 1', images=None, tool_name=None, tool_calls=None) logprobs=None


#### Now we are able to process user input for images

In [22]:
from pathlib import Path
import os

importlib.reload(pipeline_module)
pipeline = pipeline_module.OCRPipeline()
# Extract only the document title and section headers.
image_path = Path("images/PHYS207_PAGE_1.png")

if not image_path.exists():
    print(f"Image not found: {image_path}")
else:
    print(f"Image found: {image_path}")

# run the image through the pipeline
try:
    file_input = input("Enter the path to the image file: ")
    prompt_input = input("Enter the prompt for the vision model: ")

    print(f"Your prompt was: {prompt_input}")
    print(f"The image you selected was: {file_input}")

    # Set to None if you want unlimited output from the model.
    # max_output_tokens = 200

    print("Processing image...")
    print("Method vars:", pipeline.process_image.__code__.co_varnames[:4])
    result = pipeline.process_image(
        f"images/{file_input}",
        prompt_input,
        max_output_tokens=None,
        think=False,
    )

    txt_path = pipeline.save_output_text(
        f"images/{file_input}",
        result
    )

    print("Saved to:", txt_path)

    print("Final output:", result["final_output"])
    print("Done reason:", result["done_reason"])
    print("Total duration:", result["total_duration"])
    print("Prompt eval count:", result["prompt_eval_count"])
    print("Eval count:", result["eval_count"])

except Exception as e:
    print("An error occurred:", str(e))


Image found: images/PHYS207_PAGE_1.png
Your prompt was: give a short description of all the diagrams or pictures you see on the page
The image you selected was: PHYS207_PAGE_1.png
Processing image...
Method vars: ('self', 'image_path', 'prompt', 'max_output_tokens')
Saved to: extracted_texts/PHYS207_PAGE_1.txt
Final output: The image is a hand-drawn physics notes page titled “Electric Charge & Coulomb’s Law,” dated 1/12/26. It contains four main sections, each with diagrams and formulas:

---

**Section 1: Coulomb’s Law**

- A diagram shows two charged particles (one blue +, one red –) separated by distance x.
- Arrows labeled F_E point away from each charge, indicating repulsive force for like charges or attractive for opposite.
- Formula: |F_E| = α · q₁q₂ / x²  
  where α = 9·10⁹ N·m²/C² = 1/(4π ε₀)
- Notes: Force points along line connecting particles; opposites attract, likes repel.
- Electron charge and mass values listed on right.

---

**Section 2: Charges on Ropes**

- Diagram 